# 🎤 YingMusic-SVC — Google Colab

เปลี่ยนเสียงร้องในเพลงให้เป็นเสียงของคนอื่น (ใช้แค่ตัวอย่างเสียง 10–25 วินาที)

1. `Runtime → Change runtime type → T4 GPU`
2. กด ▶ ที่ Cell ด้านล่าง
3. รอติดตั้ง (ครั้งแรก ~5–8 นาที) → จะมีปุ่ม **Choose Files** ให้อัปโหลดเพลง (และเสียงเป้าหมายถ้าเลือก `upload`)
4. รอแปลงเสร็จ → ฟังผลได้ใน notebook และไฟล์จะดาวน์โหลดให้อัตโนมัติ

จะแปลงเพลงถัดไป กด ▶ ซ้ำได้เลย (ไม่ต้องติดตั้งใหม่)

> ทำงานใน notebook ทั้งหมด ไม่เปิดหน้าเว็บ — ใช้กับ Colab ฟรีได้โดยไม่ผิดนโยบาย

In [ ]:
#@title ▶ แปลงเสียงเพลง
#@markdown **เสียงเป้าหมาย** — `upload` = อัปโหลดเสียงร้อง/พูดที่สะอาด ไม่มีดนตรี ยาว 10–25 วินาที
TIMBRE = "upload"  #@param ["upload", "prompts/tp.wav", "prompts/syz.wav"]
#@markdown ติ๊กถ้าเพลงมีดนตรีประกอบ (จะแยกเสียงร้องออกก่อน) — เอาติ๊กออกถ้าเป็นไฟล์เสียงร้องล้วน
HAS_ACCOMPANIMENT = True  #@param {type:"boolean"}
#@markdown ติ๊กถ้าอยากเก็บเสียงคอรัสของต้นฉบับไว้ในดนตรีตอนมิกซ์กลับ
KEEP_BACKING_VOCAL = False  #@param {type:"boolean"}
#@markdown คุณภาพ vs ความเร็ว: 100 = ค่าแนะนำ, 30–50 = เร็วขึ้น
DIFFUSION_STEPS = 100  #@param {type:"slider", min:10, max:200, step:10}
#@markdown ---
#@markdown ขั้นสูง (ปกติไม่ต้องแก้)
REPO_URL = "https://github.com/DDME36/YingMusic-SVC"  #@param {type:"string"}
BRANCH = ""  #@param {type:"string"}
HF_TOKEN = ""  #@param {type:"string"}

import os, shlex, shutil, subprocess, time, glob
from google.colab import files
from IPython.display import Audio, display

def sh(cmd, cwd=None):
    """รันคำสั่ง, แสดง output สด, หยุดถ้าล้มเหลว"""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise SystemExit(f"❌ ล้มเหลว (exit {p.returncode}): {cmd}")

q = shlex.quote
REPO_DIR = "/content/YingMusic-SVC"
READY = "/content/.ymsvc_ready"

if subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode != 0:
    raise SystemExit("❌ ไม่พบ GPU — Runtime → Change runtime type → เลือก T4 GPU แล้วรันใหม่")

# ================= 1. ติดตั้ง (ครั้งแรกครั้งเดียว) =================
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(f"git clone -q --depth 1 {'-b ' + q(BRANCH) if BRANCH else ''} {q(REPO_URL)} {REPO_DIR}")
else:
    sh(f"git -C {REPO_DIR} pull -q --ff-only")
os.chdir(REPO_DIR)
sh('git log -1 --format="code version: %h %s"')

if not os.path.exists(READY):
    print("📦 กำลังติดตั้ง dependencies ... (ข้อความ dependency conflict ของ pip ข้ามได้)")
    sh("apt-get -qq update > /dev/null && apt-get -qq install -y sox libsox-fmt-all ffmpeg > /dev/null")
    from importlib.metadata import version
    # ใช้ torch/torchaudio ที่ Colab ติดมาแล้ว (ล็อกไว้ไม่ให้ pip เปลี่ยน)
    sh(f"pip install -q torch=={version('torch')} torchaudio=={version('torchaudio')} "
       '"librosa>=0.10.2,<1" "transformers>=4.46.3,<5" "huggingface-hub>=0.28.1,<1.0" '
       'munch einops "descript-audio-codec==1.0.0" "gradio>=5.23,<6" pydub soundfile pyyaml '
       'omegaconf ml_collections loralib wandb beartype "rotary_embedding_torch==0.3.5" matplotlib')

    print("⬇️ กำลังโหลดโมเดล ...")
    from huggingface_hub import hf_hub_download, snapshot_download
    token = HF_TOKEN or None
    hf_hub_download("GiantAILab/YingMusic-SVC", "YingMusic-SVC-full.pt", local_dir="checkpoints", token=token)
    hf_hub_download("GiantAILab/YingMusic-SVC", "bs_roformer.ckpt", local_dir="accom_separation/ckpt/bs_roformer", token=token)
    # โมเดลเสริม: cache path เดียวกับที่ my_inference.py / hf_utils.py ใช้
    hf_hub_download("lj1995/VoiceConversionWebUI", "rmvpe.pt", cache_dir="./checkpoints")
    hf_hub_download("funasr/campplus", "campplus_cn_common.bin", cache_dir="./checkpoints")
    snapshot_download("nvidia/bigvgan_v2_44khz_128band_512x", cache_dir="./checkpoints/hf_cache",
                      allow_patterns=["config.json", "bigvgan_generator.pt"])
    snapshot_download("openai/whisper-small", cache_dir="./checkpoints/hf_cache",
                      allow_patterns=["*.json", "*.safetensors", "*.txt"])

    sh("python -c \"import torch, torchaudio, librosa, transformers, dac; "
       "print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())\"")
    open(READY, "w").close()
    print("✅ ติดตั้งเสร็จ")

SVC_CKPT = os.path.abspath("checkpoints/YingMusic-SVC-full.pt")
SEP_CKPT = os.path.abspath("accom_separation/ckpt/bs_roformer/bs_roformer.ckpt")
SEP_CFG = os.path.abspath("accom_separation/ckpt/bs_roformer/config_bd_roformer.yaml")

# ================= 2. อัปโหลดไฟล์ =================
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
WORK = os.path.abspath(f"colab_runs/{RUN_ID}")
os.makedirs(WORK, exist_ok=True)

def upload_one(prompt):
    print(prompt)
    up = files.upload()
    if not up:
        raise SystemExit("❌ ไม่ได้อัปโหลดไฟล์")
    name = next(iter(up))
    dst = os.path.join(WORK, "upload_" + os.path.basename(name).replace(" ", "_"))
    shutil.move(name, dst)
    return dst, os.path.splitext(os.path.basename(name))[0]

song_raw, song_title = upload_one("🎵 อัปโหลดเพลงต้นฉบับ (mp3/wav/flac/m4a):")
SONG_WAV = os.path.join(WORK, "song.wav")
sh(f"ffmpeg -hide_banner -loglevel error -y -i {q(song_raw)} -ar 44100 {q(SONG_WAV)}")

if TIMBRE == "upload":
    timbre_raw, _ = upload_one("🗣️ อัปโหลดเสียงเป้าหมาย (เสียงร้อง/พูดสะอาด 10–25 วินาที):")
    TIMBRE_WAV = os.path.join(WORK, "timbre.wav")
    sh(f"ffmpeg -hide_banner -loglevel error -y -i {q(timbre_raw)} -ar 44100 -ac 1 {q(TIMBRE_WAV)}")
else:
    TIMBRE_WAV = os.path.abspath(TIMBRE)

# ================= 3. แยกเสียงร้อง / ดนตรี =================
if HAS_ACCOMPANIMENT:
    print("🎚️ กำลังแยกเสียงร้องออกจากดนตรี ...")
    sep_in, sep_out = os.path.join(WORK, "sep_in"), os.path.join(WORK, "sep_out")
    os.makedirs(sep_in, exist_ok=True)
    shutil.copy(SONG_WAV, sep_in)
    sh(f"PYTHONWARNINGS=ignore python inference.py --model_type bs_roformer --config_path {q(SEP_CFG)} "
       f"--start_check_point {q(SEP_CKPT)} --input_folder {q(sep_in)} --store_dir {q(sep_out)} --disable_detailed_pbar",
       cwd="accom_separation")
    stems = os.path.join(sep_out, "song")
    SOURCE_VOCAL = os.path.join(stems, "vocals.wav")
    ACCOMPANY = os.path.join(stems, "instrumental.wav")
    if KEEP_BACKING_VOCAL:
        mixed_acc = os.path.join(stems, "instrumental_with_backing.wav")
        sh(f"ffmpeg -hide_banner -loglevel error -y -i {q(ACCOMPANY)} -i {q(os.path.join(stems, 'backing_vocal.wav'))} "
           f"-filter_complex amix=inputs=2:normalize=0 {q(mixed_acc)}")
        ACCOMPANY = mixed_acc
else:
    SOURCE_VOCAL, ACCOMPANY = SONG_WAV, ""

# ================= 4. แปลงเสียง =================
print("🎤 กำลังแปลงเสียง ... (เพลงยาว 3–4 นาทีบน T4 ใช้เวลาหลายนาที)")
EXP = f"colab_{RUN_ID}"
sh(f"python -u my_inference.py --source {q(SOURCE_VOCAL)} --target {q(TIMBRE_WAV)} "
   f"--diffusion-steps {int(DIFFUSION_STEPS)} --checkpoint {q(SVC_CKPT)} --expname {EXP} --cuda 0 --fp16 True "
   f"--accompany {q(ACCOMPANY)} --config configs/YingMusic-SVC.yml")

vc_files = sorted(glob.glob(f"outputs/{EXP}/*.wav"))
mix_files = sorted(glob.glob(f"outputs/{EXP}/accompany/*.wav"))
if not vc_files:
    raise SystemExit("❌ ไม่พบไฟล์ผลลัพธ์ — ดู error ด้านบน")

# ================= 5. ฟัง & ดาวน์โหลด =================
results = [("เสียงร้องที่แปลงแล้ว", vc_files[-1], f"{song_title}_vocal.wav")]
if mix_files:
    results.insert(0, ("เพลงที่มิกซ์กับดนตรีแล้ว", mix_files[-1], f"{song_title}_cover.wav"))
for label, wav, nice_name in results:
    final = os.path.join(WORK, nice_name.replace(" ", "_"))
    shutil.copy(wav, final)
    preview = os.path.splitext(final)[0] + ".mp3"  # preview เป็น mp3 ไม่ให้ notebook หนัก
    sh(f"ffmpeg -hide_banner -loglevel error -y -i {q(final)} -b:a 192k {q(preview)}")
    print(f"✅ {label}: {final}")
    display(Audio(filename=preview))
    files.download(final)